In [0]:
df_kafka_raw = spark.read.json(
    "s3://finstream-data-ingestion/raw/stream_transactions"
)

display(df_kafka_raw)

account_id,amount,currency,description,merchant_id,payment_method,transaction_id,transaction_status,transaction_timestamp,transaction_type
acc_00002655,2781.99,AED,Ducimus maxime in reprehenderit.,mer_001751,bank_transfer,stream_txn_000000000003,completed,2026-08-21T15:21:09.836757+00:00,purchase
acc_00008903,11989.42,EUR,Necessitatibus itaque iure vero voluptate nam neque.,null,card,stream_txn_000000000008,completed,2026-08-21T15:21:14.870614+00:00,deposit
acc_00010792,34719.22,AED,Fugit repudiandae ad minima dolorum exercitationem praesentium.,null,card,stream_txn_000000000009,completed,2026-08-21T15:21:15.878212+00:00,deposit
acc_00010065,1184.98,INR,Harum nemo dignissimos sapiente eaque atque sint.,mer_001096,bank_transfer,stream_txn_000000000014,completed,2026-08-21T15:21:20.914347+00:00,purchase
acc_00011971,18404.46,SGD,Omnis harum fugiat sit consequuntur.,mer_001451,online,stream_txn_000000000018,completed,2026-08-21T15:21:24.935915+00:00,purchase
acc_00012775,86059.72,AED,Nemo nesciunt eos voluptates consectetur.,null,bank_transfer,stream_txn_000000000019,completed,2026-08-21T15:21:25.941135+00:00,deposit
acc_00013108,170402.17,SGD,Corrupti voluptate voluptas quas.,null,bank_transfer,stream_txn_000000000020,completed,2026-08-21T15:21:26.947130+00:00,deposit
acc_00013605,149510.37,GBP,Tenetur exercitationem ipsum.,null,cash,stream_txn_000000000023,completed,2026-08-21T15:21:29.961886+00:00,deposit
acc_00012704,28124.44,EUR,Quis repudiandae molestiae.,mer_001144,upi,stream_txn_000000000024,completed,2026-08-21T15:21:30.970608+00:00,purchase
acc_00010129,20199.7,SGD,Corrupti saepe suscipit hic quibusdam possimus.,null,bank_transfer,stream_txn_000000000025,completed,2026-08-21T15:21:31.976155+00:00,transfer


In [0]:
df_kafka_raw.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- description: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- transaction_timestamp: string (nullable = true)
 |-- transaction_type: string (nullable = true)



In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_timestamp
)

df_bronze = (
    df_kafka_raw
    .withColumn(
        "transaction_timestamp",
        to_timestamp(col("transaction_timestamp"))
    )
    .withColumn(
        "_ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_path")
    )
    .select(
        "transaction_id",
        "account_id",
        "merchant_id",
        "transaction_timestamp",
        "transaction_type",
        "amount",
        "currency",
        "transaction_status",
        "payment_method",
        "description",
        "_ingestion_timestamp",
        "_source_file"
    )
)

In [0]:
df_bronze.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- description: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)



In [0]:
display(df_bronze.limit(10))

transaction_id,account_id,merchant_id,transaction_timestamp,transaction_type,amount,currency,transaction_status,payment_method,description,_ingestion_timestamp,_source_file
stream_txn_000000000003,acc_00002655,mer_001751,2026-08-21T15:21:09.836Z,purchase,2781.99,AED,completed,bank_transfer,Ducimus maxime in reprehenderit.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000008,acc_00008903,null,2026-08-21T15:21:14.870Z,deposit,11989.42,EUR,completed,card,Necessitatibus itaque iure vero voluptate nam neque.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000009,acc_00010792,null,2026-08-21T15:21:15.878Z,deposit,34719.22,AED,completed,card,Fugit repudiandae ad minima dolorum exercitationem praesentium.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000014,acc_00010065,mer_001096,2026-08-21T15:21:20.914Z,purchase,1184.98,INR,completed,bank_transfer,Harum nemo dignissimos sapiente eaque atque sint.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000018,acc_00011971,mer_001451,2026-08-21T15:21:24.935Z,purchase,18404.46,SGD,completed,online,Omnis harum fugiat sit consequuntur.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000019,acc_00012775,null,2026-08-21T15:21:25.941Z,deposit,86059.72,AED,completed,bank_transfer,Nemo nesciunt eos voluptates consectetur.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000020,acc_00013108,null,2026-08-21T15:21:26.947Z,deposit,170402.17,SGD,completed,bank_transfer,Corrupti voluptate voluptas quas.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000023,acc_00013605,null,2026-08-21T15:21:29.961Z,deposit,149510.37,GBP,completed,cash,Tenetur exercitationem ipsum.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000024,acc_00012704,mer_001144,2026-08-21T15:21:30.970Z,purchase,28124.44,EUR,completed,upi,Quis repudiandae molestiae.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl
stream_txn_000000000025,acc_00010129,null,2026-08-21T15:21:31.976Z,transfer,20199.7,SGD,completed,bank_transfer,Corrupti saepe suscipit hic quibusdam possimus.,2026-08-23T13:16:02.820Z,s3://finstream-data-ingestion/raw/stream_transactions/stream_transactions_20260822_200011_147901.jsonl


In [0]:
print("New Kafka/S3 records:", df_bronze.count())

New Kafka/S3 records: 63


In [0]:
existing_count = spark.table(
    "finstream_data_pipeline.bronze.transactions"
).count()

print("Existing Bronze records:", existing_count)

Existing Bronze records: 100063


In [0]:
bronze_ids = spark.table(
    "finstream_data_pipeline.bronze.transactions"
).select("transaction_id")

new_records = (
    df_bronze
    .join(
        bronze_ids,
        on="transaction_id",
        how="left_anti"
    )
)

print("New unique records:", new_records.count())

New unique records: 0


In [0]:
display(
    new_records
    .select(
        "transaction_id",
        "account_id",
        "transaction_type",
        "amount",
        "currency",
        "transaction_status",
        "transaction_timestamp"
    )
    .limit(20)
)

transaction_id,account_id,transaction_type,amount,currency,transaction_status,transaction_timestamp


In [0]:
new_records.write \
    .mode("append") \
    .saveAsTable(
        "finstream_data_pipeline.bronze.transactions"
    )

In [0]:
bronze = spark.table(
    "finstream_data_pipeline.bronze.transactions"
)

print("Bronze row count:", bronze.count())

Bronze row count: 100063


In [0]:
display(
    bronze
    .filter(col("_source_file").contains("raw/transactions"))
    .select(
        "transaction_id",
        "account_id",
        "merchant_id",
        "transaction_timestamp",
        "transaction_type",
        "amount",
        "currency",
        "transaction_status",
        "payment_method",
        "_ingestion_timestamp",
        "_source_file"
    )
    .orderBy(col("_ingestion_timestamp").desc())
    .limit(20)
)

transaction_id,account_id,merchant_id,transaction_timestamp,transaction_type,amount,currency,transaction_status,payment_method,_ingestion_timestamp,_source_file
stream_txn_000000000005,acc_00008629,mer_000053,2026-08-21T15:21:11.849Z,payment,569.38,USD,completed,cash,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000002,acc_00012909,null,2026-08-21T15:31:27.158Z,transfer,81882.15,AED,completed,upi,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000006,acc_00004940,null,2026-08-21T15:21:12.858Z,transfer,45580.63,AED,completed,bank_transfer,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000007,acc_00013565,null,2026-08-21T15:31:32.183Z,withdrawal,20048.29,GBP,completed,online,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000002,acc_00007203,mer_000971,2026-08-21T15:21:08.830Z,payment,42766.64,AED,completed,online,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000007,acc_00008758,null,2026-08-21T15:21:13.863Z,withdrawal,40411.41,AED,completed,cash,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000026,acc_00007776,null,2026-08-21T15:21:32.983Z,withdrawal,36932.13,USD,completed,online,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000015,acc_00000431,null,2026-08-21T15:21:21.919Z,transfer,44165.38,GBP,completed,cash,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000029,acc_00010685,null,2026-08-21T15:21:36.011Z,transfer,77004.16,SGD,completed,cash,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
stream_txn_000000000012,acc_00003571,mer_000704,2026-08-21T15:31:37.205Z,payment,3925.29,EUR,completed,online,2026-08-22T19:13:34.476Z,s3://finstream-data-ingestion/raw/transactions/2026/08/21/transactions_20260821_200456_655638.jsonl
